In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
import shutil

In [21]:
from config import config

In [22]:
root = Path("../../..")

In [23]:
SOURCE_DIR = Path("D:/MSc/3. Spring 2026/CSE715/Project/MERGE_Bimodal_Balanced/lyrics")
DEST_DIR = root / config.LYRICS_DIR_EN
METADATA_FILE = root / config.METADATA_DIR / "metadata_en.csv"
FEATURES_DIR = root / config.FEATURES_DIR

In [24]:
SOURCE_DIR.exists(), METADATA_FILE.exists(), FEATURES_DIR.exists() # have to be True

(True, True, True)

In [25]:
DEST_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
all_feature_files = list(FEATURES_DIR.glob("*.npy"))
parent_ids = [f.stem.split('_')[0] for f in all_feature_files]
unique_song_ids = sorted(list(set(parent_ids)))

print(f"Total .npy files found: {len(all_feature_files)}")
print(f"Total unique songs extracted: {len(unique_song_ids)}")
print("First 10 unique IDs:", unique_song_ids[:10])

Total .npy files found: 1770
Total unique songs extracted: 155
First 10 unique IDs: ['', '-Bfa-aVa7MU', '-ph4mykFp9I', '8dLOs8LK4OQ', 'A061-66', 'A120-168', 'A151-172', 'A167', 'A168', 'BWIW9aBNmlk']


In [27]:
df_meta = pd.read_csv(METADATA_FILE)
relevant_meta = df_meta[df_meta['audio_file_stem'].isin(unique_song_ids)]

In [28]:
len(relevant_meta["audio_file_stem"].unique())

125

In [29]:
relevant_meta.head()

,audio_file_stem,lyric_file_stem,quadrant,genres
0,MT0009968194,MT0009968194,Q1,Country
1,MT0004506984,MT0004506984,Q1,"Adult Contemporary,R&B,Contemporary Pop/Rock,D..."
2,MT0011358600,MT0011358600,Q4,Blues
3,A168,L168,Q4,"Electronic,Ethnic Fusion,Club/Dance,New Age"
4,MT0009239443,MT0009239443,Q2,"International,Pop/Rock"


In [30]:
def find_lyrics_file(quadrant, stem):
    """
    Find the .txt files in "<SOURCE_DIR>/<quadrant>/<stem>.txt".
    """
    candidate = SOURCE_DIR / quadrant / f"{stem}.txt"
    return candidate if candidate.exists() else None

In [31]:
def transfer_lyrics_file(src, dst=DEST_DIR, copy_instead_of_move=True):
    """
    Creates destination folder if not already present. Copies or moves the .txt file from the source folder to the destination folder.
    """
    shutil.copy2(src, dst) if copy_instead_of_move else shutil.move(str(src), str(dst))

In [32]:
missing = []
found = 0
for _, row in relevant_meta.iterrows():
    stem = str(row["lyric_file_stem"]).strip()
    quadrant = str(row["quadrant"]).strip()
    try:
        src_file = find_lyrics_file(quadrant=quadrant, stem=stem)
        if src_file is None: 
            missing.append(stem)
            continue
        transfer_lyrics_file(src=src_file)
        found += 1
    except Exception as e:
        print(f"{e}: {stem}")

In [33]:
missing, found

([], 125)